# Example: An MCP Server for the National Weather Service API
In this example, we turn our software development kit (SDK) for the National Weather Service (NWS) API into a Model Context Protocol (MCP) server. The server offers two tools: `get_location` finds the nearest city and the NWS forecast office of a location, and `get_hourly_forecast` gets the hourly forecast. We act as the host and send the requests that a host would send for its large language model (LLM).

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Start and discover an MCP server:__ A host starts a stdio server as a subprocess, and sends it one line of JSON text per message. We'll start the weather server from Julia, and read the request and the response that discover the server.
> * __List and call the tools of a server:__ Each tool has a name, a description, and an input schema. We'll list the two tools of the weather server, and call each tool with arguments that match its schema.
> * __Tell a protocol error from a tool execution error:__ A request that the server cannot process gets a [JSON-RPC](https://www.jsonrpc.org/specification) error, while a tool that fails returns a normal result with `isError` set to `true`. We'll cause both kinds of errors and read each error response.

Let's get started!

___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this example's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The code in the [src](src) folder has three parts:

* __Weather SDK:__ [Types.jl](src/Types.jl), [Factory.jl](src/Factory.jl), [Network.jl](src/Network.jl), and [Handler.jl](src/Handler.jl) define the data types, build the request URLs, make the HTTP requests, and process the responses.
* __Server:__ [Tools.jl](src/Tools.jl) defines the weather tools, and [Server.jl](src/Server.jl) reads MCP messages and writes the responses.
* __Client:__ The functions in [Client.jl](src/Client.jl) start the server and send requests.

The [server.jl](server.jl) file is the program that a host starts. It loads the code, builds the weather server, and runs the server loop.

### Constants
We use the latitude and longitude of Ithaca, NY. We start the server with the same Julia installation and the same course project as this notebook:


In [2]:
latitude = 42.443961; # latitude of Ithaca, NY (units: decimal degrees, north is positive)
longitude = -76.501881; # longitude of Ithaca, NY (units: decimal degrees, east is positive)
julia_program = joinpath(Sys.BINDIR, "julia"); # the Julia program that runs this notebook
server_command = `$(julia_program) --startup-file=no --project=$(CHEME5800_ROOT) $(CHEME5800_L13C_SERVER)`; # the command that starts the weather server

___
## Task 1: Start the Server and Discover It
In this task, we start the weather server as a subprocess, and ask the server which protocol versions it supports and what it offers.

[The `connect(...)` function](src/Client.jl) runs the server command, and connects to the standard input and the standard output of the server. It returns [a `MyMCPClientModel` instance](src/Types.jl). We pass `verbose = true`, so the client prints every message that it sends and receives.

Let's start the server:

In [3]:
client = connect(server_command; verbose = true); # start the server, and connect to it

Next, we send a `server/discover` request with [the `discover(...)` function](src/Client.jl). The first response takes a few seconds, because the server loads its packages before it reads its first message.

> __What is happening behind the scenes?__ [The `send_request(...)` function](src/Client.jl) adds the `_meta` object to the parameters of the request. It writes the request as one line of JSON text to the standard input of the server. It then reads one line from the standard output of the server, and parses that line.

We store the response in the `discover_response` variable:

In [4]:
discover_response = discover(client); # send a server/discover request, and read the response

Client → Server
{
  "id": 1,
  "jsonrpc": "2.0",
  "method": "server/discover",
  "params": {
    "_meta": {
      "io.modelcontextprotocol/clientCapabilities": {},
      "io.modelcontextprotocol/clientInfo": {
        "name": "cheme-5800-notebook",
        "version": "1.0.0"
      },
      "io.modelcontextprotocol/protocolVersion": "2026-07-28"
    }
  }
}
Server → Client
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "cacheScope": "public",
    "capabilities": {
      "tools": {}
    },
    "instructions": "Weather data for locations in the United States from the National Weather Service. Give each location as a latitude and a longitude in decimal degrees.",
    "resultType": "complete",
    "supportedVersions": [
      "2026-07-28"
    ],
    "ttlMs": 3600000
  }
}


__What do we see?__ The request names protocol [version `2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28) in its `_meta` object. The response has the same `id` as the request:

* The `supportedVersions` field says that the server supports protocol [version `2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28).
* The `capabilities` field says that the server offers tools.
* The `_meta` object gives the name of the server, `cheme-5800-weather`, and its version.

We stop printing every message, because the later messages are long:

In [5]:
client.verbose = false; # do not print every message from here on

___
## Task 2: List the Tools
In this task, we ask the server for its tools, and read the input schema of one tool.

Let's send a `tools/list` request with [the `listtools(...)` function](src/Client.jl), and put the name, the required arguments, and the description of each tool into a table:

In [6]:
list_response = listtools(client); # send a tools/list request, and read the response
tools = list_response["result"]["tools"]; # one dictionary for each tool

# one row for each tool -
tool_table = DataFrame(name = String[], required = String[], description = String[]);
for tool ∈ tools
    required_arguments = join(tool["inputSchema"]["required"], ", "); # the arguments that every call must give
    push!(tool_table, (tool["name"], required_arguments, tool["description"]));
end
pretty_table(tool_table;
    backend = :text,
    alignment = :l,
    auto_wrap = true,
    fixed_data_column_widths = [20, 20, 44], # column widths in characters: name, required arguments, description
    display_size = (-1, -1), # show every row
    table_format = TextTableFormat(borders = text_table_borders__compact)
);


 ---------------------- ---------------------- ----------------------------------------------
  name                   required               description                                  
  String                 String                 String                                       
 ---------------------- ---------------------- ----------------------------------------------
  get_hourly_forecast    latitude, longitude    Get the hourly weather forecast for a
                                                location in the United States from the
                                                National Weather Service: temperature, wind,
                                                and a short forecast for each hour.
  get_location           latitude, longitude    Find the nearest city, the state, and the
                                                National Weather Service forecast office of
                                                a location in the United States.
 ----------

The server has two tools, and both tools require a latitude and a longitude. An LLM reads the description of each tool to choose the tool for a question.

Next, let's find the hourly forecast tool in the list, and print its input schema:

In [7]:
forecast_tool = nothing;
for tool ∈ tools
    if (tool["name"] == "get_hourly_forecast")
        forecast_tool = tool;
    end
end
println(JSON.json(forecast_tool["inputSchema"]; pretty = true)) # the JSON Schema of the arguments

{
  "properties": {
    "hours": {
      "description": "Number of hours to return, starting now (default: 12)",
      "maximum": 48,
      "minimum": 1,
      "type": "integer"
    },
    "latitude": {
      "description": "Latitude in decimal degrees, north is positive, e.g., 42.444 for Ithaca, NY",
      "maximum": 90,
      "minimum": -90,
      "type": "number"
    },
    "longitude": {
      "description": "Longitude in decimal degrees, east is positive, e.g., -76.502 for Ithaca, NY",
      "maximum": 180,
      "minimum": -180,
      "type": "number"
    }
  },
  "required": [
    "latitude",
    "longitude"
  ],
  "type": "object"
}


__What do we see?__ The schema lists three arguments. The `latitude` and `longitude` arguments are required numbers. The `hours` argument is an optional integer from `1` to `48`.

___
## Task 3: Call the Tools
In this task, we call each tool as a host would for its LLM. Then, we cause a protocol error and a tool execution error.

> __Code walkthrough:__ Let's open [Tools.jl](src/Tools.jl) in VS Code. The `get_hourly_forecast` tool pairs a name and an input schema with a handler, the Julia function that runs the tool. The handler calls our NWS SDK.

Let's call the `get_location` tool for Ithaca, NY with [the `calltool(...)` function](src/Client.jl), and print the response of the server:

In [8]:
location_arguments = Dict{String,Any}("latitude" => latitude, "longitude" => longitude); # the arguments of the call
location_response = calltool(client, "get_location", location_arguments);
println(JSON.json(location_response; pretty = true)) # the response of the server

{
  "id": 3,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "content": [
      {
        "text": "{\"city\":\"Ithaca\",\"forecast_office\":\"BGM\",\"grid_x\":44,\"grid_y\":70,\"state\":\"NY\",\"time_zone\":\"America/New_York\"}",
        "type": "text"
      }
    ],
    "isError": false,
    "resultType": "complete",
    "structuredContent": {
      "city": "Ithaca",
      "forecast_office": "BGM",
      "grid_x": 44,
      "grid_y": 70,
      "state": "NY",
      "time_zone": "America/New_York"
    }
  }
}


__What do we see?__ The `isError` flag of the result is `false`, because the tool ran without an error. The result holds the data in two forms:

* The `structuredContent` object holds the result as JSON data. The NWS forecast office for Ithaca, NY is `BGM`, the office in Binghamton, NY.
* The `content` array holds the same result as text. A host gives this text to its LLM.

> __Prediction:__ If we change `hours` from `6` to `3` in the next cell, what changes in the response? Make the change and rerun the cell to check your prediction.

Let's call the hourly forecast tool and put the returned `periods` into a table:

In [9]:
forecast_arguments = Dict{String,Any}("latitude" => latitude, "longitude" => longitude, "hours" => 6);
forecast_response = calltool(client, "get_hourly_forecast", forecast_arguments);
forecast_result = forecast_response["result"]["structuredContent"]; # the result as JSON data

# one row for each hour of the forecast -
forecast_table = DataFrame(start_time = String[], temperature = Int64[], wind = String[], short_forecast = String[]);
for period ∈ forecast_result["periods"]
    wind = period["wind_speed"] * " " * period["wind_direction"]; # e.g., 10 mph NW
    push!(forecast_table, (period["start_time"], period["temperature"], wind, period["short_forecast"]));
end
pretty_table(forecast_table; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

 --------------------------- ------------- ----------- ----------------------------
                 start_time   temperature        wind               short_forecast 
                     String         Int64      String                       String 
 --------------------------- ------------- ----------- ----------------------------
  2026-11-15T07:00:00-05:00            35    7 mph NW   Slight Chance Rain Showers
  2026-11-15T08:00:00-05:00            35    8 mph NW   Slight Chance Rain Showers
  2026-11-15T09:00:00-05:00            38    9 mph NW   Slight Chance Rain Showers
  2026-11-15T10:00:00-05:00            41   10 mph NW   Slight Chance Rain Showers
  2026-11-15T11:00:00-05:00            42   12 mph NW   Slight Chance Rain Showers
  2026-11-15T12:00:00-05:00            43   12 mph NW   Slight Chance Rain Showers
 --------------------------- ------------- ----------- ----------------------------


The temperatures are in degrees Fahrenheit. A host gives the forecast text to its LLM, which can then answer a question such as "Will it rain in Ithaca in the next six hours?"

### Two Kinds of Errors
First, let's call a tool that the server does not have:

In [10]:
unknown_response = calltool(client, "get_snowfall", location_arguments); # the server has no get_snowfall tool
println(JSON.json(unknown_response; pretty = true))

{
  "error": {
    "code": -32602,
    "message": "Unknown tool: get_snowfall"
  },
  "id": 5,
  "jsonrpc": "2.0"
}


__What do we see?__ The response holds an `error` object, and no `result`. The code `-32602` means invalid params: the request named a tool that the server does not have. This protocol error usually means a bug in the host.

Next, let's ask for the forecast in Paris, France. The request is valid, but the NWS covers only the United States:

In [11]:
paris_arguments = Dict{String,Any}("latitude" => 48.8566, "longitude" => 2.3522); # Paris, France
paris_response = calltool(client, "get_hourly_forecast", paris_arguments);
println(JSON.json(paris_response; pretty = true))

{
  "id": 6,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "content": [
      {
        "text": "ArgumentError: HTTP GET call failed with status code 404",
        "type": "text"
      }
    ],
    "isError": true,
    "resultType": "complete"
  }
}


__What do we see?__ The response holds a normal `result`, with `isError` set to `true`. The tool ran, and the NWS API answered with status code `404`, i.e., not found. A host gives the text of this tool execution error to its LLM, and the LLM can tell the user that the location is outside the United States.

### Close the Connection
We are done with the server. [The `close(...)` function](src/Client.jl) closes the standard input of the server. The server loop then ends, and the server exits.

To rerun a cell above after the server exits, first rerun the cell that calls [the `connect(...)` function](src/Client.jl) to start a new server. Let's close the connection:

In [12]:
close(client); # close the standard input of the server, and wait for the server to exit

> __Connect a host to the weather server:__ A host such as Claude Code starts the same server with the same command. For example, the command below registers the weather server with Claude Code, where `/path/to/L13c/server.jl` is the path of the [server.jl](server.jl) file on your computer:
>
> `claude mcp add weather -- julia --startup-file=no /path/to/L13c/server.jl`
>
> Claude Code then lists the tools of the server. It asks for your approval before its LLM calls a tool, unless your permission settings already allow the call.

___

## Summary
In this example, we started an MCP server that wraps our NWS SDK, called its two tools as a host would, and caused a protocol error and a tool execution error.

> __Key Takeaways:__
>
> * __A stdio server:__ We started the weather server as a subprocess and exchanged one line of JSON text per message with it. Each of our requests named the protocol version, and each response repeated the `id` of its request.
> * __The tools call our SDK:__ Each tool checks its arguments and gets the weather data through our NWS SDK. Each successful result holds the data twice, as text for an LLM and as JSON data for a program.
> * __Two kinds of errors:__ We caused a protocol error by naming a tool that the server does not have, and a tool execution error by asking for a forecast outside the United States. A protocol error usually means a bug in the host or the server, while the text of a tool execution error tells the LLM what went wrong.

Any MCP host that supports protocol [version `2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28) can now use our weather server, with no code written for that host.

___